In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "halimajaved592/urdu-english-code-switching-xlm-roberta"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("✅ Model loaded successfully!")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: halimajaved592/urdu-english-code-switching-xlm-roberta
Key                        | Status     | 
---------------------------+------------+-
classifier.weight          | UNEXPECTED | 
classifier.bias            | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded successfully!


In [3]:
print("Model class:", type(model).__name__)
print("\nModel configuration:")
print(model.config)

print("\nLabel mapping:")
print(model.config.id2label)

print("\nNumber of labels:", model.config.num_labels)

Model class: XLMRobertaForSequenceClassification

Model configuration:
XLMRobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "XLMRobertaForTokenClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "URD",
    "1": "ENG",
    "2": "MIX"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "ENG": 1,
    "MIX": 2,
    "URD": 0
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.15.0",
  "type_vocab_size": 1,
  "use_cache": false,
  "vocab_size": 250002
}


Label mapping:
{0: '

In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "halimajaved592/urdu-english-code-switching-xlm-roberta"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

print("✅ Token classification model loaded successfully!")
print("Labels:", model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Token classification model loaded successfully!
Labels: {0: 'URD', 1: 'ENG', 2: 'MIX'}


In [5]:
import torch

sentence = "Aaj mera mood nahi hai for anything"

inputs = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=-1)[0]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, prediction in zip(tokens, predictions):
    print(f"{token:20} → {model.config.id2label[prediction.item()]}")

<s>                  → URD
▁Aaj                 → URD
▁mera                → URD
▁mood                → ENG
▁nahi                → URD
▁hai                 → URD
▁for                 → ENG
▁anything            → ENG
</s>                 → URD


In [6]:
test_sentences = [
    "Aaj mera mood nahi hai for anything",
    "Mujhe kal university jana hai",
    "I have a meeting tomorrow",
    "Yaar kal presentation hai still not ready",
    "Bhai assignment submit kar di I am relieved",
    "This is a beautiful day"
]

for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    print("\nSentence:", sentence)
    print("-" * 60)

    for token, prediction in zip(tokens, predictions):
        if token not in ["<s>", "</s>"]:
            label = model.config.id2label[prediction.item()]
            print(f"{token:20} → {label}")


Sentence: Aaj mera mood nahi hai for anything
------------------------------------------------------------
▁Aaj                 → URD
▁mera                → URD
▁mood                → ENG
▁nahi                → URD
▁hai                 → URD
▁for                 → ENG
▁anything            → ENG

Sentence: Mujhe kal university jana hai
------------------------------------------------------------
▁Mujh                → URD
e                    → URD
▁kal                 → URD
▁university          → ENG
▁jana                → URD
▁hai                 → URD

Sentence: I have a meeting tomorrow
------------------------------------------------------------
▁I                   → ENG
▁have                → ENG
▁a                   → ENG
▁meeting             → ENG
▁tomorrow            → ENG

Sentence: Yaar kal presentation hai still not ready
------------------------------------------------------------
▁Yaar                → URD
▁kal                 → URD
▁presentation        → ENG
▁hai       

In [7]:
test_sentence = "Yaar kal presentation hai still not ready"

inputs = tokenizer(
    test_sentence,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predictions = torch.argmax(probabilities, dim=-1)[0]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Sentence: {test_sentence}")
print("-" * 70)
print(f"{'Token':20} {'Label':10} {'Confidence':10}")
print("-" * 70)

for i, token in enumerate(tokens):
    if token not in ["<s>", "</s>"]:
        label = model.config.id2label[predictions[i].item()]
        confidence = probabilities[0, i, predictions[i]].item()

        print(f"{token:20} {label:10} {confidence:.2%}")

Sentence: Yaar kal presentation hai still not ready
----------------------------------------------------------------------
Token                Label      Confidence
----------------------------------------------------------------------
▁Yaar                URD        99.91%
▁kal                 URD        99.92%
▁presentation        ENG        97.27%
▁hai                 URD        99.91%
▁still               ENG        97.18%
▁not                 ENG        98.76%
▁ready               ENG        98.89%
